In [3]:
from typing import Tuple

# string_similarity.py

def _levenshtein_distance(a: str, b: str) -> int:
    """Compute Levenshtein distance (iterative, O(min(len(a),len(b))) memory)."""
    if a == b:
        return 0
    len_a, len_b = len(a), len(b)
    if len_a == 0:
        return len_b
    if len_b == 0:
        return len_a
    # Ensure a is the shorter string to use less memory
    if len_a > len_b:
        a, b = b, a
        len_a, len_b = len_b, len_a

    previous = list(range(len_a + 1))
    for j in range(1, len_b + 1):
        bj = b[j - 1]
        current = [j] + [0] * len_a
        for i in range(1, len_a + 1):
            cost = 0 if a[i - 1] == bj else 1
            current[i] = min(
                previous[i] + 1,      # deletion
                current[i - 1] + 1,   # insertion
                previous[i - 1] + cost  # substitution
            )
        previous = current
    return previous[len_a]

def string_similarity(s1: str, s2: str,
                      case_sensitive: bool = False,
                      ignore_whitespace: bool = False) -> float:
    """
    Return a normalized similarity score between 0.0 and 1.0 (1.0 = exact match).
    Uses normalized Levenshtein distance: score = 1 - (levenshtein / max_len).
    Parameters:
    - s1, s2: input strings
    - case_sensitive: if False, comparison is case-insensitive
    - ignore_whitespace: if True, all whitespace is removed before comparison
    """
    if not case_sensitive:
        s1 = s1.lower()
        s2 = s2.lower()
    if ignore_whitespace:
        s1 = "".join(s1.split())
        s2 = "".join(s2.split())

    if s1 == s2:
        return 1.0
    if len(s1) == 0 or len(s2) == 0:
        # one empty and the other not -> no similarity
        return 0.0

    dist = _levenshtein_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    score = 1.0 - (dist / max_len)
    # Guard against tiny numerical drift
    if score < 0.0:
        return 0.0
    if score > 1.0:
        return 1.0
    return score

# Example usage (can be removed when used as a module):
if __name__ == "__main__":
    pairs = [
        ("hello", "hello"),
        ("Hello", "hello"),
        ("kitten", "sitting"),
        ("", "nonempty"),
        (" ", ""),
    ]
    for a, b in pairs:
        print(f"'{a}' vs '{b}': {string_similarity(a, b, case_sensitive=True)}")

'hello' vs 'hello': 1.0
'Hello' vs 'hello': 0.8
'kitten' vs 'sitting': 0.5714285714285714
'' vs 'nonempty': 0.0
' ' vs '': 0.0
